# Week 3 skills notebook: types, units and missing values

This notebook belongs to the skills track. It uses a made-up shop's purchase book and a made-up list of monthly price changes, so the skills stay separate from this week's project.

You will see that a value means what its type says, turn numbers stored as text into numbers, put every column in one unit, compound percentage changes, and find missing values in three disguises. Three tasks are yours, and each one has a CHECK cell.

**How to use it:** run each cell in order with the play button or Shift+Enter. Read the output before you move on.

## 1. The same characters, different meanings

A computer stores every value as a pattern of bits. The type of a value is the agreement that says how to read that pattern. The same characters can be text or a number, and the result of a calculation depends on which one Python sees.

In [ ]:
print('"7" + "7" gives', "7" + "7")                 # two pieces of text, joined
print("7 + 7 gives", 7 + 7)                         # two numbers, added
print('int("7") + int("7") gives', int("7") + int("7"))   # text turned into numbers first
print("The character 7 is stored as code", ord("7"))

A type also has a size. A column of small whole numbers can be stored in 8 bits, and the next cell prints the range those 8 bits can hold. A result that does not fit wraps around to the start of the range, and no error appears.

In [ ]:
import numpy as np

print("an 8-bit column holds", np.iinfo(np.uint8).min, "to", np.iinfo(np.uint8).max)
small = np.array([200], dtype=np.uint8)
print("200 + 100 in an 8-bit column:", (small + 100)[0])
print("200 + 100 in a 64-bit column:", (small.astype(np.int64) + 100)[0])

Most data tools store whole numbers in 64 bits, so this rarely happens by accident. It does happen when a file or a program picks a small type to save space. The wrong answer then looks like an ordinary number.

## 2. Load a file where numbers arrive as text

Spreadsheets set to Dutch, German or French write a comma as the decimal mark and put a semicolon between the columns. The cell below builds the text of such a file for a made-up shop's purchase book, then loads it the way you would load any file.

The cell also keeps the true values in a table called `truth`, so that the CHECK cells can test your work. A real file never arrives with its truth.

In [ ]:
import io
import pandas as pd

rng = np.random.default_rng(3)
n = 40
per_kg = {"rice": 2.5, "flour": 2.0, "sugar": 2.2, "cooking oil": 6.0}   # made-up prices per kilogram
truth = pd.DataFrame({
    "date": pd.date_range("2026-03-02", periods=n, freq="D"),
    "item": rng.choice(list(per_kg), size=n),
    "bags": rng.integers(1, 9, size=n).astype(float),
    "bag_kg": rng.choice([0.5, 1.0, 2.0, 5.0, 25.0], size=n),
})
truth["unit_price"] = (truth["bag_kg"] * truth["item"].map(per_kg) * rng.uniform(0.9, 1.1, size=n)).round(2)
truth.loc[[4, 9, 15, 21, 33], "bags"] = np.nan        # five purchases where nobody counted the bags

# the file as a spreadsheet set to Dutch would save it
bags_text = truth["bags"].map(lambda b: "" if pd.isna(b) else str(int(b)))
bags_text[9] = "unknown"
bags_text[[15, 33]] = "-1"
size_text = truth["bag_kg"].map(lambda kg: f"{kg * 1000:.0f} g" if kg < 1 else f"{kg:.0f} kg")
price_text = truth["unit_price"].map(lambda p: f"{p:.2f}".replace(".", ","))
lines = ["date;item;bags;unit_price;bag_size"] + [
    f"{d:%Y-%m-%d};{i};{b};{p};{s}"
    for d, i, b, p, s in zip(truth["date"], truth["item"], bags_text, price_text, size_text)
]
file_text = "\n".join(lines)

purchases = pd.read_csv(io.StringIO(file_text), sep=";")
purchases.head(12)

## 3. Describe the types

The four questions from week 1 still come first. This time, look hard at the third one: what type does each column hold? pandas marks a column of text as `object`. A column of numbers that pandas reads as text cannot be averaged until you convert it.

In [ ]:
print("rows and columns:", purchases.shape)
print()
print(purchases.dtypes)
print()
print("values in the bags column:", sorted(purchases["bags"].dropna().unique()))
print("first prices as written:", purchases["unit_price"].head(3).tolist())

## 4. Read code an AI assistant wrote

We asked an AI assistant: *"Turn the unit_price column into numbers and give me the average price per bag."* It wrote the cell below. Run it, then decide whether the answer is right.

In [ ]:
# Written by an AI assistant
price_ai = purchases["unit_price"].str.replace(",", "").astype(float)
print("Average price per bag:", round(price_ai.mean(), 2))

The code runs without an error, and the answer is still wrong. The file uses a comma as the decimal mark, and the assistant's code deleted every comma, as if each one separated thousands.

The next cell reads the first row by hand, converts the column the right way and compares the two averages.

In [ ]:
price = purchases["unit_price"].str.replace(",", ".").astype(float)   # the comma is the decimal mark
print("Row 0 as written:", purchases.loc[0, "unit_price"], "-> the AI's number:", price_ai[0], "-> read correctly:", price[0])
print("Average price per bag, read correctly:", round(price.mean(), 2))
print("The AI's answer was too large by a factor of:", round(price_ai.mean() / price.mean(), 1))

# CHECK: every converted price must match the true price
assert np.allclose(price, truth["unit_price"])
print("CHECK passed")

A better fix tells pandas the file's agreement when it loads the file. `sep=";"` names the separator between columns, and `decimal=","` names the decimal mark.

In [ ]:
again = pd.read_csv(io.StringIO(file_text), sep=";", decimal=",")
print(again.dtypes)
print("same average, read at load time:", round(again["unit_price"].mean(), 2))

## 5. Missing values in three disguises

A missing value does not always look like an empty cell. In this file it comes in three disguises: an empty cell, a word (`unknown`) and a code number (`-1`, which the shop writes when nobody counted the bags). The first step is to find all three.

`pd.to_numeric(..., errors="coerce")` turns anything that is not a number into a missing value, and it does so without a word. List the values it will drop before you rely on it.

In [ ]:
bags_raw = purchases["bags"]
as_number = pd.to_numeric(bags_raw, errors="coerce")
print("empty cells:", int(bags_raw.isna().sum()))
print("words that are not numbers:", sorted(bags_raw[as_number.isna() & bags_raw.notna()].unique()))
print("rows with the code -1:", int((as_number == -1).sum()))

The code `-1` is the most dangerous disguise, because it is a number and pandas averages it without complaint. Compare the average with the code left in and with the code treated as missing.

In [ ]:
bags = as_number.where(as_number != -1)          # -1 means nobody counted, so it is missing too
print("Average bags per purchase with -1 left in:", round(as_number.mean(), 2))
print("Average bags per purchase with -1 treated as missing:", round(bags.mean(), 2))
print("purchases with a count:", int(bags.count()), "of", len(bags))

# CHECK: the cleaned counts must match the truth, and be missing where the truth is missing
assert (bags.isna() == truth["bags"].isna()).all()
assert np.allclose(bags.dropna(), truth["bags"].dropna())
print("CHECK passed")

## 6. Keep a cleaning log

Every change you make to a table changes the answers it gives. A cleaning log lists each change and its reason, so a reader can see exactly how the raw file became the table you used. The cell below starts a clean table and its log.

In [ ]:
clean = pd.DataFrame({"date": pd.to_datetime(purchases["date"]), "item": purchases["item"]})
log = []

clean["unit_price"] = price
log.append("unit_price: read the comma as the decimal mark and stored the prices as numbers")
clean["bags"] = bags
log.append(f"bags: turned empty cells, 'unknown' and the code -1 into missing values; {int(bags.isna().sum())} of {len(bags)} purchases have no count")

for line in log:
    print("-", line)

## 7. Your turn: one unit per column

The `bag_size` column mixes two units. Some bags are written in grams, such as `500 g`, and some in kilograms, such as `25 kg`. A column with two units cannot be added or compared.

Ask your AI assistant for code that adds a column `bag_kg` to `clean` with every bag size in kilograms, and adds one line to `log`. Paste its code into the cell below, run it, then run the CHECK cell. Before you trust it, convert one value in grams by hand.

In [ ]:
# your turn 1: add clean["bag_kg"] (every bag size in kilograms) and one line to log

In [ ]:
# CHECK 1
assert "bag_kg" in clean, "clean has no bag_kg column yet"
assert np.allclose(clean["bag_kg"], truth["bag_kg"]), "some bag sizes are not in kilograms"
assert len(log) >= 3, "add one line to the cleaning log"
print("CHECK 1 passed: bag sizes run from", clean["bag_kg"].min(), "to", clean["bag_kg"].max(), "kg")
for line in log:
    print("-", line)

## 8. Your turn: a price in the right unit

A price per bag cannot compare a 500 g bag with a 25 kg bag. A price per kilogram can. Store in `price_per_kg` the average price per kilogram of each item, one value per item, computed from `clean`.

Your AI assistant may suggest `groupby`. Ask it to explain that line in words before you run it.

In [ ]:
# your turn 2: price_per_kg = the average of unit_price / bag_kg for each item
price_per_kg = None

In [ ]:
# CHECK 2
expected = (truth["unit_price"] / truth["bag_kg"]).groupby(truth["item"]).mean()
assert price_per_kg is not None, "price_per_kg is still empty"
assert np.allclose(price_per_kg.sort_index(), expected.sort_index())
print("CHECK 2 passed: average price per kilogram")
print(price_per_kg.sort_index().round(2))

## 9. Percent, fraction and times

A percentage change is a unit too. A rise of 55 percent multiplies a price by 1 plus 55 divided by 100. That multiplier is called the growth factor. A price that becomes 2.5 times as high has risen by much more than 2.5 percent.

The function below turns a percentage change into its growth factor. The last two lines show why the difference matters.

In [ ]:
def factor(pct):
    """A percentage change as the number a price is multiplied by."""
    return 1 + pct / 100

print("55 percent as a growth factor:", factor(55))
print("2.5 times as high, as a percentage change:", (2.5 - 1) * 100)
print("A rise of 20 percent two years in a row:", round(factor(20) * factor(20), 2))
print("A rise of 50 percent and then a fall of 50 percent:", factor(50) * factor(-50))

## 10. Compound, do not add

The cell below makes a made-up list of monthly price changes for a basket of goods over 24 months. The change for month 1 compares the basket's price with the month before the list starts. Two months were never measured.

Over several months the changes multiply, because each month's change applies to a price that earlier months already raised. Adding the percentages ignores that, so the sum falls short when prices keep rising.

In [ ]:
rng = np.random.default_rng(11)
change = pd.Series(np.round(rng.normal(3.5, 2.5, size=24), 1),
                   index=pd.RangeIndex(1, 25, name="month"), name="pct_change")
change.loc[[17, 18]] = np.nan            # two months nobody measured
print("missing months:", change[change.isna()].index.tolist())

year1 = change.loc[1:12]
print("Year 1, monthly changes added:", round(year1.sum(), 1), "percent")
print("Year 1, monthly changes compounded:", round((factor(year1).prod() - 1) * 100, 1), "percent")

Compounding means applying each change in turn and nothing more. The loop below starts from a price of 100 and applies the twelve changes of year 1 one after another. Its result must match the compounded figure.

In [ ]:
price_level = 100.0
for pct in year1:
    price_level = price_level * factor(pct)
print("A price of 100 after year 1:", round(price_level, 2))

# CHECK: the loop and the product give the same answer
assert abs(price_level - 100 * factor(year1).prod()) < 1e-9
print("CHECK passed")

## 11. A gap breaks the chain

When one link in a chain of changes is missing, the total is unknown. pandas does not tell you so. Its `sum()` and `prod()` skip missing values, which amounts to assuming that nothing changed in the missing month. With `skipna=False`, pandas returns a missing value instead.

The small example below has four months, and month 2 is missing.

In [ ]:
example = pd.Series([10.0, np.nan, 10.0, 10.0], index=[1, 2, 3, 4])
print("changes:", example.tolist())
print("added, pandas default:", example.sum())
print("compounded, pandas default:", round((factor(example).prod() - 1) * 100, 1), "percent")
print("compounded, skipna=False:", factor(example).prod(skipna=False))
print("months 3 and 4, which have no gap:", round((factor(example.loc[3:4]).prod() - 1) * 100, 1), "percent")

The honest answer has two parts. Over a stretch with no gap, you can say exactly how much prices changed. Across the gap, you can only say what an assumption would give, and you must name the assumption. If prices did not fall in month 2, for example, the four months together rose by at least the compounded figure that pandas printed by default.

## 12. Chart the gap

A bar chart of the monthly changes shows the two missing months as empty places. A label on each one says so, so that no reader mistakes an empty place for a month without change.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 3))
ax.bar(change.index, change)
for m in change[change.isna()].index:
    ax.text(m, 0.3, "no data", rotation=90, ha="center", va="bottom", fontsize=8)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_title("Monthly price change of a made-up basket (two months not measured)")
ax.set_xlabel("month")
ax.set_ylabel("change (percent)")
plt.tight_layout()
plt.show()

## 13. Your turn: the year with a gap

Year 2 runs from month 13 to month 24, and months 17 and 18 are missing. Ask your AI assistant for code that computes two numbers, paste it below and run the CHECK.

- `year2_total` is the total price change over year 2, in percent, computed so that the answer is missing because of the gap.
- `year2_after` is the price change over months 19 to 24, in percent. That stretch has no gap.

An assistant may well write code that returns a number for `year2_total`. If yours does, the CHECK fails, and the failure is the lesson. Write down in one sentence what the assistant's number assumed about months 17 and 18.

In [ ]:
# your turn 3: compute year2_total and year2_after, both in percent
year2_total = None
year2_after = None

In [ ]:
# CHECK 3
assert year2_after is not None, "year2_after is still empty"
assert year2_total is not None, "year2_total is still empty: it should be a missing value, not None"
assert np.isnan(year2_total), "year2_total hides the two missing months; try prod(skipna=False)"
assert abs(year2_after - (factor(change.loc[19:24]).prod() - 1) * 100) < 1e-6
print("CHECK 3 passed: year 2 as a whole is unknown, and months 19 to 24 changed by", round(year2_after, 1), "percent")

## 14. What you did

You read numbers that arrived as text, gave every column one unit, compounded percentage changes instead of adding them, found missing values in three disguises and refused a total the data could not support. Every change to the purchase book went into the log, so a reader can follow each one.